#Accounts SQL

In [0]:
df_accounts = spark.sql(
    """
    SELECT
        row_number() OVER(ORDER BY account_id) AS account_sk,
        *
    FROM nedbank_ovation.silver.accounts
    """
)

df_accounts = df_accounts.withColumnRenamed("customer_ref", "customer_id")

df_accounts.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .saveAsTable("nedbank_ovation.gold.dim_accounts")

#Customers SQL trans

In [0]:
df_customers = spark.sql(
    """
        WITH customer_age AS (
            SELECT
            row_number() OVER(ORDER BY customer_id) AS customer_sk,
            floor((date_diff(CAST(ingestion_time AS DATE), dob)) / 365.25) AS age,
            * EXCEPT (dob)
            FROM nedbank_ovation.silver.customers
        )

        SELECT
        *,
        CASE
            WHEN age >= 65 THEN "65+"
            WHEN age >= 56 THEN "56-65"
            WHEN age >= 46 THEN "46-55"
            WHEN age >= 36 THEN "36-45"
            WHEN age >= 26 THEN "26-35"
            WHEN age >= 18 THEN "18-25"
        ELSE NULL
        END AS age_band
        FROM customer_age
    """
)

df_customers.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .saveAsTable("nedbank_ovation.gold.dim_customers")


#Trnsaction SQL trans

In [0]:
df_transactions = spark.sql(
    """
        SELECT
            row_number() OVER(ORDER BY transaction_id) AS transaction_sk,
            acc.account_sk,
            cust.customer_sk,
            TO_TIMESTAMP(concat(transaction_date, " ", transaction_time), "yyyy-MM-dd HH:mm:ss") AS transaction_timestamp,
            t.* EXCEPT (transaction_time)
        FROM nedbank_ovation.silver.transactions AS t
        JOIN nedbank_ovation.gold.dim_accounts AS acc
            ON t.account_id = acc.account_id
        JOIN nedbank_ovation.gold.dim_customers AS cust
            ON acc.customer_id = cust.customer_id

    """
)

df_transactions.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .saveAsTable("nedbank_ovation.gold.fact_transactions")


#combined gold tables


In [0]:
import yaml

with open("../config/pipeline_config.yaml") as pc:
    config = yaml.safe_load(pc)

#dim_accounts
df_accounts = spark.sql(
    f"""
    SELECT
        row_number() OVER(ORDER BY account_id) AS account_sk,
        *
    FROM delta.`{config["output"]["silver_path"]}/accounts`
    """
)
df_accounts = df_accounts.withColumnRenamed("customer_ref", "customer_id")

#dim_customers
df_customers = spark.sql(
    """
        WITH customer_age AS (
            SELECT
            row_number() OVER(ORDER BY customer_id) AS customer_sk,
            floor((date_diff(CAST(ingestion_time AS DATE), dob)) / 365.25) AS age,
            * EXCEPT (dob)
            FROM delta.`{config["output"]["silver_path"]}/customers`
        )

        SELECT
        *,
        CASE
            WHEN age >= 65 THEN "65+"
            WHEN age >= 56 THEN "56-65"
            WHEN age >= 46 THEN "46-55"
            WHEN age >= 36 THEN "36-45"
            WHEN age >= 26 THEN "26-35"
            WHEN age >= 18 THEN "18-25"
        ELSE NULL
        END AS age_band
        FROM customer_age
    """
)

#transactions
df_transactions = spark.sql(
    f"""
        SELECT
            row_number() OVER(ORDER BY transaction_id) AS transaction_sk,
            acc.account_sk,
            cust.customer_sk,
            TO_TIMESTAMP(concat(transaction_date, " ", transaction_time), "yyyy-MM-dd HH:mm:ss") AS transaction_timestamp,
            * EXCEPT (transaction_time)
        FROM delta.`{config["output"]["silver_path"]}/transactions`
        JOIN delta.`{config["output"]["gold_path"]}/dim_accounts` AS acc
            ON transactions.account_id = acc.account_id
        JOIN {config["output"]["gold_path"]}/dim_customers AS cust
            ON acc.customer_id = cust.customer_id
    """
)

data_frame = {
    "dim_accounts" : df_accounts,
    "dim_customers" : df_customers,
    "fact_transactions" : df_transactions
}

for table_name, df in data_frame.items():
    df.write\
        .mode("overwrite")\
        .format("delta")\
        .option("overwriteSchema", "true")\
        .save(f"{config["output"]["gold_path"]}/{table_name}")